In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
from tqdm import tqdm

def get_starbucks_seoul_filtered():
    options = Options()
    # options.add_argument('--headless') # 확인 후 필요하면 주석 해제하세요.
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36")
    options.add_argument('--window-size=1200,1020')

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 20)
    start_time = time.time()

    try:
        print("🚀 스타벅스 웹사이트 접속 및 검색 설정 중...")
        driver.get("https://www.starbucks.co.kr/store/store_map.do?disp=quick")
        
        # 1. 초기 로딩 대기
        wait.until(EC.invisibility_of_element_located((By.CLASS_NAME, "loading_dimm")))

        # 2. 지역 검색 -> 서울 -> 전체 클릭 (유저님이 알려주신 클래스 활용)
        loca_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".loca_search h3 a")))
        driver.execute_script("arguments[0].click();", loca_btn)
        time.sleep(1.5)

        seoul_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a.set_sido_cd_btn[data-sidocd="01"]')))
        driver.execute_script("arguments[0].click();", seoul_btn)
        time.sleep(1.5)

        all_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a.set_gugun_cd_btn[data-guguncd=""]')))
        driver.execute_script("arguments[0].click();", all_btn)
        
        # 3. 데이터 렌더링 대기
        print("⏳ 서울 전 지점 데이터를 불러오는 중...")
        wait.until(EC.invisibility_of_element_located((By.CLASS_NAME, "loading_dimm")))
        time.sleep(8) 

        # 4. 모든 매장 요소 수집 (이때는 초기 6개+서울 전체가 다 잡힐 수 있음)
        stores = driver.find_elements(By.CSS_SELECTOR, 'li.quickResultLstCon')
        
        raw_data = []
        for store in tqdm(stores, desc="데이터 파싱 중", unit="store"):
            name = store.get_attribute('data-name')
            # 주소 누락 방지를 위해 innerText 사용
            full_text = store.find_element(By.TAG_NAME, 'p').get_attribute('innerText')
            address = full_text.split('\n')[0] if '\n' in full_text else full_text
            address = address.replace("1522-3232", "").strip()

            raw_data.append({'매장명': name, '주소': address})

        # 5. 데이터프레임 생성
        df = pd.DataFrame(raw_data)

        # 6. [핵심] 주소에 '서울특별시'가 포함된 데이터만 남기기
        # 이 과정에서 초기 위치 기반 6개 지점(서울이 아닐 경우)이 제거됩니다.
        df_seoul = df[df['주소'].str.contains('서울특별시')].copy()
        
        # 인덱스 재정렬 (0부터 다시 시작하게)
        df_seoul.reset_index(drop=True, inplace=True)
        
        return df_seoul

    except Exception as e:
        print(f"❌ 에러 발생: {e}")
        return None
    finally:
        driver.quit()

# --- 실행 및 결과 출력 ---
if __name__ == "__main__":
    seoul_df = get_starbucks_seoul_filtered()

    if seoul_df is not None:
        print("\n" + "="*70)
        print(f"✨ 서울특별시 스타벅스 데이터 수집 완료 (총 {len(seoul_df)}개)")
        print("="*70)

        # 요청하신 상위 10개, 하위 10개 출력
        print("\n[ 상위 10개 매장 (Head) ]")
        print(seoul_df.head(10))

        print("\n" + "-"*70)

        print("[ 하위 10개 매장 (Tail) ]")
        print(seoul_df.tail(10))
        print("="*70)

🚀 스타벅스 웹사이트 접속 및 검색 설정 중...
⏳ 서울 전 지점 데이터를 불러오는 중...


데이터 파싱 중: 100%|██████████| 685/685 [00:26<00:00, 26.29store/s]



✨ 서울특별시 스타벅스 데이터 수집 완료 (총 679개)

[ 상위 10개 매장 (Head) ]
        매장명                          주소
0   역삼아레나빌딩     서울특별시 강남구 언주로 425 (역삼동)
1    논현역사거리    서울특별시 강남구 강남대로 538 (논현동)
2   신사역성일빌딩    서울특별시 강남구 강남대로 584 (논현동)
3    국기원사거리    서울특별시 강남구 테헤란로 125 (역삼동)
4    대치재경빌딩  서울특별시 강남구 남부순환로 2947 (대치동)
5      봉은사역    서울특별시 강남구 봉은사로 619 (삼성동)
6   압구정윤성빌딩     서울특별시 강남구 논현로 834 (신사동)
7    코엑스별마당    서울특별시 강남구 영동대로 513 (삼성동)
8  삼성역섬유센터R    서울특별시 강남구 테헤란로 518 (대치동)
9      압구정R     서울특별시 강남구 언주로 861 (신사동)

----------------------------------------------------------------------
[ 하위 10개 매장 (Tail) ]
      매장명                                      주소
669    상봉  서울특별시 중랑구 상봉로 131 (상봉동, 상봉 듀오트리스 주상복합)
670   중랑역                서울특별시 중랑구 망우로30길 3 (상봉동)
671  중랑구청                        서울특별시 중랑구 신내로 72
672  사가정역                       서울특별시 중랑구 면목로 310
673   상봉역                 서울특별시 중랑구 망우로 307 (상봉동)
674   면목역                 서울특별시 중랑구 면목로 403 (면목동)
675    묵동   서울특별시 중랑구 동일로 952 (묵동, 로프트원 태릉입구역) 1층
676   망우동   

In [ ]:
import requests
import pandas as pd
import time

def create_starbucks_seoul_fixed():
    # 1. API 엔드포인트 및 파라미터
    # r 파라미터는 세션마다 달라질 수 있으므로 최신 코드를 반영합니다.
    url = "https://www.starbucks.co.kr/store/getStore.do?r=R6B7ZV3918"
    
    payload = {
        'p_sido_cd': '01',           # 서울 코드
        'p_gugun_cd': '', 
        'ins_lat': '37.2635727',
        'ins_lng': '127.0286009',
        'new_gj': '0',
        'iend': '2000',              # 넉넉하게 설정
        'set_date': '',
        'in_biz_cd' : ''
    }
    
    # 스타벅스 서버가 '진짜 브라우저'라고 믿게 만드는 헤더
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        "Referer": "https://www.starbucks.co.kr/store/store_map.do", # 이게 없으면 0개 반환됨
        "X-Requested-With": "XMLHttpRequest"
    }

    print("📡 스타벅스 서버에 데이터를 요청 중입니다...")
    start_time = time.time()

    try:
        # POST 요청
        response = requests.post(url, data=payload, headers=headers)
        
        if response.status_code == 200:
            json_data = response.json()
            store_list = json_data.get('list', [])
            
            # 2. 데이터가 0개인지 즉시 확인
            if not store_list:
                print("⚠️ 서버에서 빈 데이터를 보냈습니다. 헤더나 세션 코드를 확인해야 합니다.")
                return None

            print(f"📦 서버 응답 수신 완료 ({len(store_list)}개 지점)")

            # 3. 요청하신 4가지 키값 추출 및 컬럼명 매칭
            refined_data = []
            for s in store_list:
                # 주소 필드가 addr 또는 doro_address에 들어있을 수 있음
                address = s.get('addr') or s.get('doro_address') or ""
                
                # '서울특별시'가 포함된 경우만 필터링
                if "서울특별시" in address:
                    refined_data.append({
                        '매장코드': s.get('s_code'),
                        '매장명': s.get('s_name'),
                        '주소': address,
                        '오픈날짜': s.get('open_dt')
                    })
            
            # 4. 데이터프레임 생성 및 CSV 저장
            df = pd.DataFrame(refined_data)
            
            if not df.empty:
                filename = "starbucks.csv"
                df.to_csv(filename, index=False, encoding='utf-8-sig')
                
                total_time = time.time() - start_time
                print("\n" + "="*60)
                print(f"✨ 수집 성공! 파일명: {filename}")
                print(f"📊 서울특별시 지점 수: {len(df)}개")
                print(f"⏱️ 총 소요 시간: {total_time:.4f}초")
                print("="*60)
                return df
            else:
                print("❌ 필터링 결과 서울 지역 매장이 0개입니다.")
        else:
            print(f"❌ 접속 실패 (상태 코드: {response.status_code})")

    except Exception as e:
        print(f"❌ 오류 발생: {e}")

if __name__ == "__main__":
    create_starbucks_seoul_fixed()

📡 스타벅스 서버에 데이터를 요청 중입니다...
📦 서버 응답 수신 완료 (679개 지점)

✨ 수집 성공! 파일명: starbucks.csv
📊 서울특별시 지점 수: 679개
⏱️ 총 소요 시간: 1.6528초


In [29]:
import requests
import pandas as pd
import time
import os

def append_starbucks_gyeonggi():
    # 1. API 접속 정보 및 파라미터 설정 (경상북도용)
    url = "https://www.starbucks.co.kr/store/getStore.do?r=6K7ESKXD51"
    
    payload = {
        'p_sido_cd': '17',           # 세종 지역 코드 (17)
        'p_gugun_cd': '', 
        'ins_lat': '36.4864669',     
        'ins_lng': '127.2508515',
        'new_gj': '0',
        'iend': '2000',              # 세종 매장 전체 수집을 위해 넉넉히 설정
        'set_date': '',
        'in_biz_cd' : ''
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        "Referer": "https://www.starbucks.co.kr/store/store_map.do",
        "X-Requested-With": "XMLHttpRequest"
    }

    file_name = "starbucks.csv"
    print(f"📡 세종 지점 데이터를 수집하여 '{file_name}'에 추가합니다...")
    
    # --- 수집 시작 시간 측정 ---
    start_time = time.time()

    try:
        # 2. 데이터 요청
        response = requests.post(url, data=payload, headers=headers)
        
        if response.status_code == 200:
            store_list = response.json().get('list', [])
            
            # 3. 데이터 정제 (매장코드, 매장명, 주소, 오픈날짜)
            refined_data = []
            for s in store_list:
                address = s.get('addr') or s.get('doro_address') or ""
                
                # 세종 지점만 필터링
                if "세종" in address:
                    refined_data.append({
                        '매장코드': s.get('s_code'),
                        '매장명': s.get('s_name'),
                        '주소': address,
                        '오픈날짜': s.get('open_dt')
                    })
            
            # 4. 데이터프레임 생성
            df_new = pd.DataFrame(refined_data)
            
            # 5. [핵심] 이어붙이기 로직 (Append)
            # 파일이 이미 있으면 header=False (중복 방지), 없으면 header=True
            is_exists = os.path.exists(file_name)
            
            df_new.to_csv(file_name, 
                          mode='a',             # 'a'는 append의 약자입니다.
                          index=False, 
                          header=not is_exists, # 파일이 없을 때만 헤더를 만듭니다.
                          encoding='utf-8-sig')
            
            # --- 수집 종료 및 시간 계산 ---
            elapsed_time = time.time() - start_time
            
            print("\n" + "="*60)
            print(f"✨ 추가 완료! '{file_name}' 파일에 세종 지점이 업데이트되었습니다.")
            print(f"📊 이번에 추가된 세종 매장 수: {len(df_new)}개")
            print(f"⏱️ 수집 소요 시간: {elapsed_time:.4f}초")
            print("="*60)
            
            return df_new
        else:
            print(f"❌ 서버 응답 실패 (상태 코드: {response.status_code})")
            
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

if __name__ == "__main__":
    append_starbucks_gyeonggi()

📡 세종 지점 데이터를 수집하여 'starbucks.csv'에 추가합니다...

✨ 추가 완료! 'starbucks.csv' 파일에 세종 지점이 업데이트되었습니다.
📊 이번에 추가된 세종 매장 수: 15개
⏱️ 수집 소요 시간: 1.1064초


In [30]:
import pandas as pd
import time

# 시작 시간 측정
start_time = time.time()

# 파일 불러오기 (기존 작업 경로 기준)
file_path = r'C:\PYTHON\git-renew\starbucks.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig')

# 종료 시간 측정 및 출력
end_time = time.time()
print(f"데이터프레임 로드 소요 시간: {end_time - start_time:.4f}초")

# 불러온 데이터의 기본 정보 확인
print("\n[ 데이터 요약 정보 ]")
print(df.info())

# 상위 5개 데이터 출력
print("\n[ 데이터 샘플 ]")
print(df.head())

데이터프레임 로드 소요 시간: 0.0770초

[ 데이터 요약 정보 ]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2124 entries, 0 to 2123
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   매장코드    2124 non-null   int64 
 1   매장명     2124 non-null   object
 2   주소      2124 non-null   object
 3   오픈날짜    2124 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 66.5+ KB
None

[ 데이터 샘플 ]
   매장코드      매장명                          주소      오픈날짜
0  1509  역삼아레나빌딩  서울특별시 강남구 역삼동 721-13 아레나빌딩  20190613
1  1434   논현역사거리    서울특별시 강남구 논현동 142-2 정일빌딩  20181123
2  1595  신사역성일빌딩     서울특별시 강남구 논현동 18-4 성일빌딩  20191219
3  1527   국기원사거리   서울특별시 강남구 역삼동 648-22 동찬빌딩  20190731
4  1468   대치재경빌딩      서울특별시 강남구 대치동 599 대원빌딩  20190214


In [34]:
import pandas as pd

# 1. 기존에 수집한 starbucks.csv 파일 불러오기
file_path = 'starbucks.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig')

# 2. 필요한 컬럼만 선택 (매장코드, 매장명, 주소, 오픈날짜)
# 혹시라도 다른 컬럼이 섞여있을 경우를 대비해 딱 4개만 필터링합니다.
df = df[['매장코드', '매장명', '주소', '오픈날짜']].copy()

# 3. [핵심] '오픈날짜' 시계열 데이터로 변환
# format='%Y%m%d' (20231024 -> 2023-10-24)
df['오픈날짜'] = pd.to_datetime(df['오픈날짜'], format='%Y%m%d', errors='coerce')

# 4. 데이터 정제 확인
print("="*50)
print("✨ 최종 데이터프레임 구성 완료 (찐찐찐!)")
print("="*50)

# 컬럼 타입 확인 (오픈날짜가 datetime64인지 확인하세요)
print("\n[ 1. 컬럼 타입 정보 ]")
print(df.info())

# 데이터 샘플 확인
print("\n[ 2. 데이터 샘플 (상위 5개) ]")
print(df.head())

# 날짜 변환 실패(NaT)가 있는지 확인
null_dates = df['오픈날짜'].isna().sum()
if null_dates > 0:
    print(f"\n⚠️ 주의: 날짜 변환에 실패한 데이터가 {null_dates}건 있습니다.")
else:
    print("\n✅ 모든 오픈날짜가 성공적으로 시계열 데이터로 변환되었습니다.")

✨ 최종 데이터프레임 구성 완료 (찐찐찐!)

[ 1. 컬럼 타입 정보 ]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2124 entries, 0 to 2123
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   매장코드    2124 non-null   int64         
 1   매장명     2124 non-null   object        
 2   주소      2124 non-null   object        
 3   오픈날짜    0 non-null      datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 66.5+ KB
None

[ 2. 데이터 샘플 (상위 5개) ]
   매장코드      매장명                          주소 오픈날짜
0  1509  역삼아레나빌딩  서울특별시 강남구 역삼동 721-13 아레나빌딩  NaT
1  1434   논현역사거리    서울특별시 강남구 논현동 142-2 정일빌딩  NaT
2  1595  신사역성일빌딩     서울특별시 강남구 논현동 18-4 성일빌딩  NaT
3  1527   국기원사거리   서울특별시 강남구 역삼동 648-22 동찬빌딩  NaT
4  1468   대치재경빌딩      서울특별시 강남구 대치동 599 대원빌딩  NaT

⚠️ 주의: 날짜 변환에 실패한 데이터가 2124건 있습니다.


In [33]:
import pandas as pd
import time
import os

def finalize_starbucks_csv(file_path):
    if not os.path.exists(file_path):
        print(f"❌ '{file_path}' 경로에 파일이 없습니다. 경로를 확인해주세요.")
        return

    # --- 시작 시간 측정 ---
    start_time = time.time()
    print(f"📂 '{file_path}' 파일을 불러와 시계열 데이터로 변환 중...")

    try:
        # 1. 파일 로드
        df = pd.read_csv(file_path, encoding='utf-8-sig')

        # 2. [핵심] '오픈날짜' 시계열(datetime) 데이터로 변환
        # 기존 8자리 숫자(20231024)를 날짜 객체로 변환합니다.
        df['오픈날짜'] = pd.to_datetime(df['오픈날짜'], format='%Y%m%d', errors='coerce')

        # 3. 요청하신 4개 컬럼만 최종 선택 (매장코드, 매장명, 주소, 오픈날짜)
        df = df[['매장코드', '매장명', '주소', '오픈날짜']]

        # 4. 동일 파일에 덮어쓰기 (찐찐찐 반영)
        df.to_csv(file_path, index=False, encoding='utf-8-sig')

        # --- 종료 시간 측정 ---
        elapsed = time.time() - start_time

        print("\n" + "="*60)
        print(f"✨ 반영 완료! 'starbucks.csv'가 시계열 데이터로 업데이트되었습니다.")
        print(f"📅 데이터 예시: {df['오픈날짜'].iloc[0]} (타입: {df['오픈날짜'].dtype})")
        print(f"⏱️ 총 소요 시간: {elapsed:.4f}초")
        print("="*60)
        
        # 상위 5개 데이터 미리보기
        print("\n[ 최종 데이터프레임 샘플 ]")
        print(df.head())

    except Exception as e:
        print(f"❌ 오류 발생: {e}")

if __name__ == "__main__":
    # 유저님의 작업 경로에 맞춰 실행하세요
    target_path = "starbucks.csv" 
    finalize_starbucks_csv(target_path)

📂 'starbucks.csv' 파일을 불러와 시계열 데이터로 변환 중...

✨ 반영 완료! 'starbucks.csv'가 시계열 데이터로 업데이트되었습니다.
📅 데이터 예시: 2019-06-13 00:00:00 (타입: datetime64[ns])
⏱️ 총 소요 시간: 0.0566초

[ 최종 데이터프레임 샘플 ]
   매장코드      매장명                          주소       오픈날짜
0  1509  역삼아레나빌딩  서울특별시 강남구 역삼동 721-13 아레나빌딩 2019-06-13
1  1434   논현역사거리    서울특별시 강남구 논현동 142-2 정일빌딩 2018-11-23
2  1595  신사역성일빌딩     서울특별시 강남구 논현동 18-4 성일빌딩 2019-12-19
3  1527   국기원사거리   서울특별시 강남구 역삼동 648-22 동찬빌딩 2019-07-31
4  1468   대치재경빌딩      서울특별시 강남구 대치동 599 대원빌딩 2019-02-14


In [36]:
import pandas as pd
from sqlalchemy import create_engine, text, INT, VARCHAR, DATE
import time

def upload_starbucks_to_mysql():
    # 1. MySQL 접속 정보 설정 (본인의 환경에 맞게 수정하세요)
    host = "localhost"
    user = "root"       # 본인의 MySQL 유저네임
    password = "1234"  # 본인의 MySQL 비밀번호
    
    # DB 연결 엔진 생성 (먼저 서버에 접속)
    engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/")

    # 2. 데이터베이스 및 테이블 생성 로직
    with engine.connect() as conn:
        # 데이터베이스 생성
        conn.execute(text("CREATE DATABASE IF NOT EXISTS coffee_store;"))
        conn.execute(text("USE coffee_store;"))
        
        # 기존 테이블이 있다면 삭제 (초기화용)
        conn.execute(text("DROP TABLE IF EXISTS starbucks;"))
        
        # [요청 반영] 테이블 스키마 생성
        # 매장코드(INT), 매장명/주소(VARCHAR), 오픈날짜(DATE)
        create_table_query = """
        CREATE TABLE starbucks (
            매장코드 INT,
            매장명 VARCHAR(255),
            주소 VARCHAR(500),
            오픈날짜 DATE
        );
        """
        conn.execute(text(create_table_query))
        conn.commit()
        print("✅ 'coffee_store' DB 및 'starbucks' 테이블 생성 완료!")

    # 3. CSV 데이터 로드 및 시계열 변환
    file_path = 'starbucks.csv'
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    df['오픈날짜'] = pd.to_datetime(df['오픈날짜']) # 확실하게 시계열로 변환

    # 4. [핵심] DB에 데이터 밀어넣기
    # 다시 'coffee_store' DB 전용 엔진 생성
    db_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/coffee_store")
    
    start_time = time.time()
    
    # dtype 설정을 통해 DB 컬럼 타입을 강제 매칭합니다.
    df.to_sql(
        name='starbucks', 
        con=db_engine, 
        if_exists='append', 
        index=False,
        dtype={
            '매장코드': INT,
            '매장명': VARCHAR(255),
            '주소': VARCHAR(500),
            '오픈날짜': DATE
        }
    )
    
    elapsed = time.time() - start_time

    print("\n" + "="*60)
    print(f"✨ 데이터 삽입 성공! MySQL에서 확인 가능합니다.")
    print(f"📊 삽입된 총 데이터: {len(df)}개")
    print(f"⏱️ 삽입 소요 시간: {elapsed:.4f}초")
    print("="*60)

if __name__ == "__main__":
    upload_starbucks_to_mysql()

✅ 'coffee_store' DB 및 'starbucks' 테이블 생성 완료!

✨ 데이터 삽입 성공! MySQL에서 확인 가능합니다.
📊 삽입된 총 데이터: 2124개
⏱️ 삽입 소요 시간: 0.3924초


In [2]:
import pandas as pd
from sqlalchemy import create_engine, text, Integer, String, Date
import time

def upload_starbucks_to_postgresql():
    # 1. PostgreSQL 접속 정보 설정 (본인의 환경에 맞게 수정하세요)
    host = "localhost"
    user = "postgres"           # 기본 유저명은 보통 postgres입니다
    password = "1234"  # 본인의 PostgreSQL 비밀번호
    port = "5432"               # 기본 포트 5432
    
    # 2. 데이터베이스 생성 (기본 postgres DB에 접속해서 수행)
    # PostgreSQL은 'IF NOT EXISTS'를 실행할 때 자동 커밋 모드가 필요합니다.
    temp_engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/postgres")
    
    with temp_engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
        # DB가 있는지 확인 후 생성
        exists = conn.execute(text("SELECT 1 FROM pg_database WHERE datname='coffee_store'")).fetchone()
        if not exists:
            conn.execute(text("CREATE DATABASE coffee_store"))
            print("🐘 'coffee_store' 데이터베이스를 생성했습니다.")

    # 3. 실제 작업을 위한 coffee_store DB 엔진 생성
    db_engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/coffee_store")

    # 4. CSV 데이터 로드 및 시계열 변환
    start_time = time.time()
    file_path = 'starbucks.csv'
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # 오픈날짜 시계열 데이터 확정
    df['오픈날짜'] = pd.to_datetime(df['오픈날짜'])

    # 5. [핵심] 테이블 생성 및 데이터 삽입
    # dtype 설정을 통해 요청하신 수치형, 문자형, 날짜형 타입을 강제합니다.
    try:
        df.to_sql(
            name='starbucks', 
            con=db_engine, 
            if_exists='replace', # 기존 테이블이 있으면 새로 만듦
            index=False,
            dtype={
                '매장코드': Integer,
                '매장명': String(255),
                '주소': String(500),
                '오픈날짜': Date
            }
        )
        
        elapsed = time.time() - start_time

        print("\n" + "="*60)
        print(f"✨ PostgreSQL 삽입 성공! (coffee_store.starbucks)")
        print(f"📊 삽입된 총 데이터: {len(df)}개")
        print(f"⏱️ 전체 소요 시간: {elapsed:.4f}초")
        print("="*60)
        
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

if __name__ == "__main__":
    upload_starbucks_to_postgresql()

🐘 'coffee_store' 데이터베이스를 생성했습니다.

✨ PostgreSQL 삽입 성공! (coffee_store.starbucks)
📊 삽입된 총 데이터: 2124개
⏱️ 전체 소요 시간: 0.4064초
